# 🌳 Unidad 3 · Clase 2 — Algoritmos de Estructura: Árboles y Ensembles
### Árboles de Decisión · Random Forest · Comparación de modelos

---

## Contexto de la clase

En la clase anterior entrenaste **Regresión Lineal** y **Regresión Logística** — modelos que aprenden a través de ecuaciones matemáticas lineales.

Hoy aprenderás dos algoritmos que funcionan de manera completamente diferente: **toman decisiones en cascada**, como un árbol de preguntas de sí/no.

---

## ¿Qué son los algoritmos de estructura?

Son modelos que aprenden **particionando el espacio de datos** con reglas del tipo:

```
¿study_hours_per_day > 3.5?
    ├── SÍ → ¿mental_health_rating > 6?
    │           ├── SÍ → 🟢 Predice: Aprueba
    │           └── NO → 🔴 Predice: Reprueba
    └── NO → 🔴 Predice: Reprueba
```

Cada pregunta divide los datos en grupos más puros. El modelo aprende cuáles preguntas hacen y en qué orden.

---

## ¿Qué construiremos hoy?

| Algoritmo | Tipo | Idea central |
|---|---|---|
| **Árbol de Decisión** | Clasificación y Regresión | Un árbol de preguntas secuenciales que dividen los datos |
| **Random Forest** | Ensemble (conjunto de árboles) | Muchos árboles, cada uno entrenado con datos distintos, votando juntos |

---

## Dataset — Student Habits & Performance (continuación)

El mismo dataset de 1 000 estudiantes. Hoy lo usaremos para:
- **Clasificación:** predecir si el estudiante aprueba (`aprueba = 1`) o reprueba (`aprueba = 0`)
- **Comparar** estos nuevos modelos contra la Regresión Logística de la clase anterior

| Variable clave | Descripción |
|---|---|
| `study_hours_per_day` | Feature más correlacionada con el puntaje (r = 0.83) |
| `mental_health_rating` | Segunda feature más importante |
| `exam_score` | Target original (numérico) |
| `aprueba` | Target de clasificación que crearemos (1 si score ≥ 60) |

> ⏱️ Duración estimada: **~2 horas**

---
## 📦 Sección 1 — Importaciones

Además de las librerías habituales, hoy importamos herramientas específicas para árboles y ensembles, y para visualizar la estructura interna de un árbol de decisión.

In [ ]:
import pandas as pd                    # manipulación de DataFrames
import numpy as np                     # operaciones numéricas sobre arrays
import matplotlib.pyplot as plt        # motor base de visualización
import seaborn as sns                  # gráficas estadísticas de alto nivel

# ── Modelos de estructura ────────────────────────────────────────────────────
# DecisionTreeClassifier: árbol de decisión para clasificación binaria o multiclase
# RandomForestClassifier: ensemble de árboles con votación por mayoría
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier

# ── Preprocesamiento y evaluación ────────────────────────────────────────────
# train_test_split: divide el dataset en entrenamiento y prueba de forma estratificada
from sklearn.model_selection import train_test_split, cross_val_score

# OrdinalEncoder: codifica variables categóricas con orden (Poor < Fair < Good)
from sklearn.preprocessing import OrdinalEncoder

# Métricas para evaluar modelos de clasificación
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

# LogisticRegression: la importamos para comparar con los nuevos modelos
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

---
## 📂 Sección 2 — Carga y preparación del dataset

Repetimos el mismo pipeline de preparación de la clase anterior para que los modelos de hoy sean comparables con la Regresión Logística que ya entrenamos.

> **Principio de comparación justa:** para comparar dos modelos deben entrenarse con exactamente los mismos datos y evaluarse sobre exactamente el mismo conjunto de prueba.

In [ ]:
# Lee el CSV desde disco
df = pd.read_csv('student_habits_performance.csv')

# Elimina student_id: es un identificador único, no aporta información predictiva
df = df.drop(columns=['student_id'])

# ── Crear el target de clasificación ────────────────────────────────────────
# exam_score >= 60 define si el estudiante aprueba (1) o reprueba (0)
# Esta es la misma regla que usamos en la clase anterior
df['aprueba'] = (df['exam_score'] >= 60).astype(int)

# ── Tratar nulos ─────────────────────────────────────────────────────────────
# parental_education_level tiene 91 nulos (~9%) → imputamos con la moda
moda_edu = df['parental_education_level'].mode()[0]   # .mode()[0] toma el valor más frecuente
df['parental_education_level'] = df['parental_education_level'].fillna(moda_edu)

# ── Encoding de variables categóricas ────────────────────────────────────────
# OrdinalEncoder para variables con orden real
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)

oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)

# Encoding binario para variables Yes/No
df['part_time_job'] = (df['part_time_job'] == 'Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] == 'Yes').astype(int)

# One-Hot Encoding para gender (variable nominal sin orden)
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)

# Vista rápida del dataset ya preparado
df.head(4)

In [ ]:
# ── Definir features y targets ───────────────────────────────────────────────
# Excluimos exam_score (de ahí deriva aprueba → data leakage directo)
# Excluimos aprueba del conjunto de features (es el target)
features = ['study_hours_per_day', 'social_media_hours', 'netflix_hours',
            'attendance_percentage', 'sleep_hours', 'exercise_frequency',
            'mental_health_rating', 'part_time_job', 'diet_quality',
            'internet_quality', 'parental_education_level',
            'extracurricular_participation', 'gender_Male', 'gender_Other']

X = df[features]           # matriz de features: shape (1000, 14)
y = df['aprueba']          # vector target binario: 1 = aprueba, 0 = reprueba

# ── División estratificada ────────────────────────────────────────────────────
# stratify=y garantiza que la proporción 72%/28% se mantiene igual en train y test
# random_state=42 fija la semilla para reproducibilidad (mismo split que la clase anterior)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} estudiantes')
print(f'Prueba:        {X_test.shape[0]} estudiantes')
print(f'Proporción aprueba en train: {y_train.mean():.1%}')
print(f'Proporción aprueba en test:  {y_test.mean():.1%}')

---
## 🌳 Sección 3 — Árbol de Decisión

### ¿Qué es y cómo funciona?

Un Árbol de Decisión aprende haciendo **preguntas binarias** sobre las features.
En cada nodo del árbol elige la pregunta que **mejor separa** los casos de las dos clases.

```
Nodo raíz (la pregunta más importante)
    ├── Rama SÍ (feature ≤ umbral)
    │       └── Nodo hijo → otra pregunta
    │                 ├── Rama SÍ → Hoja (predicción final)
    │                 └── Rama NO → Hoja (predicción final)
    └── Rama NO (feature > umbral)
            └── Nodo hijo → otra pregunta...
```

### ¿Cómo elige la mejor pregunta? El criterio Gini

En cada nodo, el algoritmo prueba **todas las features** y **todos los umbrales posibles** y elige la combinación que produce los grupos más puros.

La **impureza de Gini** mide qué tan mezcladas están las clases en un nodo:

$$Gini = 1 - \sum_{k} p_k^2$$

- **Gini = 0.0** → nodo perfectamente puro (todos son clase 0 o todos son clase 1) ✅
- **Gini = 0.5** → máxima impureza (50% de cada clase)

El algoritmo elige la pregunta que **reduce más la impureza** al pasar de un nodo a sus hijos.

### El gran riesgo: overfitting

Si no limitamos la profundidad del árbol, crecerá hasta tener **un nodo hoja por cada estudiante de entrenamiento** — memorizando perfectamente los datos de entrenamiento pero fallando en datos nuevos.

El parámetro `max_depth` controla cuántos niveles de preguntas puede hacer el árbol.

### Ventaja única: interpretabilidad

A diferencia de Regresión Lineal (ecuación abstracta) o Redes Neuronales (caja negra), el Árbol de Decisión es **100% interpretable**: puedes ver exactamente qué preguntas hace y por qué llega a cada predicción.

### 3.1 — Entrenar el primer árbol (sin restricciones)

Primero lo entrenamos sin limitar la profundidad para ver el efecto del overfitting. Luego lo limitaremos.

In [ ]:
# Árbol sin restricción de profundidad: crecerá hasta memorizar los datos de entrenamiento
# criterion='gini': usa la impureza de Gini para elegir las preguntas en cada nodo
# random_state=42: reproducibilidad en los desempates cuando dos features tienen el mismo score
arbol_libre = DecisionTreeClassifier(criterion='gini', random_state=42)

# .fit() analiza los 800 estudiantes de entrenamiento y construye toda la estructura del árbol
arbol_libre.fit(X_train, y_train)

# Evaluamos sobre AMBOS conjuntos para detectar el overfitting
acc_train_libre = accuracy_score(y_train, arbol_libre.predict(X_train))  # accuracy en entrenamiento
acc_test_libre  = accuracy_score(y_test,  arbol_libre.predict(X_test))   # accuracy en prueba

print(f'Árbol sin restricción:')
print(f'  Profundidad del árbol:      {arbol_libre.get_depth()} niveles')
print(f'  Número de hojas:            {arbol_libre.get_n_leaves()}')
print(f'  Accuracy en ENTRENAMIENTO:  {acc_train_libre:.1%}')
print(f'  Accuracy en PRUEBA:         {acc_test_libre:.1%}')
print()
print(f'  Diferencia train-test: {(acc_train_libre - acc_test_libre)*100:.1f} puntos porcentuales')
print(f'  → Esta diferencia grande indica OVERFITTING: el árbol memorizó el entrenamiento.')

### 3.2 — Controlar el overfitting con `max_depth`

`max_depth` limita cuántos niveles de preguntas puede hacer el árbol.
Un árbol más profundo puede capturar más patrones pero también más ruido.

**¿Cómo elegir el mejor `max_depth`?**
Probamos varios valores y comparamos la accuracy en prueba — elegimos el que generaliza mejor.

In [ ]:
# Probamos profundidades del 1 al 15 y guardamos accuracy en train y test para cada una
profundidades   = range(1, 16)          # valores de max_depth a evaluar
acc_trains = []                          # lista para guardar accuracy de entrenamiento
acc_tests  = []                          # lista para guardar accuracy de prueba

for d in profundidades:
    # Instancia un árbol con la profundidad máxima d
    arbol_d = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=42)
    arbol_d.fit(X_train, y_train)        # entrena sobre los datos de entrenamiento

    # Calcula accuracy en ambos conjuntos y la agrega a las listas
    acc_trains.append(accuracy_score(y_train, arbol_d.predict(X_train)))
    acc_tests.append(accuracy_score(y_test,  arbol_d.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))

# Línea azul: accuracy en entrenamiento (siempre sube con más profundidad)
ax.plot(profundidades, acc_trains, marker='o', label='Entrenamiento', color='steelblue')

# Línea naranja: accuracy en prueba (sube hasta un punto y luego baja o se estanca por overfitting)
ax.plot(profundidades, acc_tests,  marker='s', label='Prueba',        color='tomato')

ax.set_title('Accuracy vs Profundidad del Árbol La brecha entre líneas = nivel de overfitting')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 — Árbol con profundidad óptima

Del gráfico anterior identificamos el `max_depth` donde la accuracy de prueba es máxima.
Ese es el árbol que mejor generaliza a estudiantes nuevos.

In [ ]:
# Encontramos automáticamente el max_depth que maximiza accuracy en el conjunto de prueba
mejor_depth = profundidades[acc_tests.index(max(acc_tests))]
print(f'Mejor max_depth según accuracy en prueba: {mejor_depth}')

# Entrenamos el árbol definitivo con ese max_depth
arbol = DecisionTreeClassifier(
    max_depth=mejor_depth,   # profundidad óptima encontrada arriba
    criterion='gini',        # criterio de impureza para elegir preguntas
    random_state=42          # semilla para reproducibilidad
)
arbol.fit(X_train, y_train)          # aprende las reglas de decisión sobre el entrenamiento

# Predicciones sobre el conjunto de prueba (datos que el árbol nunca vio)
y_pred_arbol = arbol.predict(X_test)

acc_arbol = accuracy_score(y_test, y_pred_arbol)
print(f'Accuracy del árbol óptimo en prueba: {acc_arbol:.1%}')
print(f'Profundidad real: {arbol.get_depth()} | Hojas: {arbol.get_n_leaves()}')

### 3.4 — Visualizar el árbol de decisión

Una de las grandes ventajas del Árbol de Decisión es que podemos **ver exactamente cómo decide**.

**Cómo leer cada nodo:**
- **La pregunta** (feature ≤ umbral): si es verdadera → rama izquierda; si es falsa → rama derecha
- **gini**: impureza del nodo (0 = puro, 0.5 = máxima mezcla)
- **samples**: cuántos estudiantes de entrenamiento llegaron a este nodo
- **value**: [casos clase 0, casos clase 1] en ese nodo
- **class**: predicción si el árbol se detuviera aquí
- **Color**: más intenso = más puro (un color domina claramente)

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))

# plot_tree dibuja la estructura completa del árbol aprendido
# filled=True colorea cada nodo según la clase dominante (naranja=reprueba, azul=aprueba)
# rounded=True: esquinas redondeadas (estético)
# feature_names: pone los nombres reales de las features en vez de "feature_0", "feature_1"...
# class_names: pone "Reprueba" y "Aprueba" en vez de "0" y "1"
# fontsize: tamaño de letra dentro de cada nodo
plot_tree(
    arbol,
    feature_names=features,
    class_names=['Reprueba', 'Aprueba'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title(f'Árbol de Decisión — max_depth={mejor_depth} | Accuracy={acc_arbol:.1%}', fontsize=13)
plt.tight_layout()
plt.show()

### 3.5 — Las reglas del árbol en texto plano

Además de la visualización gráfica, podemos exportar las reglas del árbol como texto.
Esto es útil para documentar el modelo o para comunicar las reglas de decisión a personas no técnicas.

In [ ]:
# export_text convierte el árbol en texto legible con indentación que muestra la jerarquía
# max_depth=3 limita la visualización a los primeros 3 niveles (los más importantes)
reglas = export_text(arbol, feature_names=features, max_depth=3)

# Imprimimos las reglas: cada nivel de indentación = un nivel más profundo del árbol
print('=== Reglas del árbol (primeros 3 niveles) ===')
print(reglas)

### 3.6 — Importancia de features del árbol

`feature_importances_` indica cuánto contribuyó cada feature a reducir la impureza Gini en todos los nodos donde fue usada.

La suma de todas las importancias siempre es 1.0. Una importancia de 0 significa que el árbol nunca usó esa feature para tomar ninguna decisión.

> **Nota:** La importancia del árbol puede sobreestimar features con muchos valores únicos (variables continuas). Random Forest corrige esto al promediar sobre muchos árboles.

In [ ]:
# feature_importances_ es un array con la importancia de cada feature en el mismo orden que X
importancias_arbol = pd.DataFrame({
    'feature':     features,
    'importancia': arbol.feature_importances_   # contribución al total de reducción de Gini
}).sort_values('importancia', ascending=False)   # ordena de más a menos importante

fig, ax = plt.subplots(figsize=(8, 5))

# Solo mostramos features con importancia > 0 (el árbol no usa todas con max_depth limitado)
imp_positivas = importancias_arbol[importancias_arbol['importancia'] > 0]
ax.barh(imp_positivas['feature'][::-1], imp_positivas['importancia'][::-1],
        color='steelblue', edgecolor='white')

ax.set_title('Importancia de features — Árbol de Decisión
(suma total = 1.0)')
ax.set_xlabel('Importancia (reducción de Gini acumulada)')
plt.tight_layout()
plt.show()

print(f'Features usadas por el árbol: {(arbol.feature_importances_ > 0).sum()} de {len(features)}')
print(f'Verificación: suma de importancias = {arbol.feature_importances_.sum():.6f}')

---
## 🌲🌲🌲 Sección 4 — Random Forest

### ¿Por qué un árbol no es suficiente?

El Árbol de Decisión tiene un problema fundamental: es **muy sensible a los datos de entrenamiento**.
Si cambias algunos ejemplos del dataset, el árbol puede cambiar completamente.
Además, para generalizar bien necesita limitarse (max_depth), perdiendo capacidad de capturar patrones complejos.

### La solución: muchos árboles votando juntos

**Random Forest** construye **N árboles de decisión** de forma paralela, cada uno entrenado con datos ligeramente diferentes, y combina sus predicciones por votación:

```
Árbol 1: → Aprueba (1)  ─────┐
Árbol 2: → Aprueba (1)       │
Árbol 3: → Reprueba (0)      ├── Votación → mayoría: Aprueba (1) ✅
Árbol 4: → Aprueba (1)       │
Árbol 5: → Aprueba (1)  ─────┘
```

### Las dos fuentes de aleatoriedad que lo hacen funcionar

#### 1. Bootstrap (muestreo con reemplazo)
Cada árbol se entrena con una **muestra aleatoria con reemplazo** del dataset de entrenamiento.
Esto significa que algunos estudiantes aparecen varias veces en la muestra de un árbol, y otros no aparecen.
Los estudiantes que no fueron seleccionados se llaman **OOB (Out-Of-Bag)** y se pueden usar para validar ese árbol sin necesidad de un conjunto de prueba separado.

#### 2. Selección aleatoria de features en cada nodo
En cada nodo, en lugar de probar **todas las features**, el árbol solo puede elegir entre una **muestra aleatoria de √n features** (donde n es el número total de features).
Esto reduce la correlación entre árboles: si `study_hours` es la mejor feature, no todos los árboles la usarán en todos los nodos, forzándolos a descubrir patrones alternativos.

### ¿Por qué esto reduce el overfitting?

La diversidad entre árboles es la clave. Los errores de cada árbol tienden a cancelarse:
- Cuando un árbol sobreajusta a un patrón de ruido, los otros árboles no lo ven
- Cuando todos los árboles coinciden en una predicción, hay alta confianza
- El promedio de muchos estimadores ruidosos converge a una estimación estable

Este principio se llama **reducción de varianza por promediado** y es la base matemática del ensemble.

### Parámetros principales

| Parámetro | Descripción | Valor típico |
|---|---|---|
| `n_estimators` | Número de árboles en el bosque | 100–500 |
| `max_depth` | Profundidad máxima de cada árbol | None (crece completo) |
| `max_features` | Features a considerar en cada nodo | `'sqrt'` (√n_features) |
| `min_samples_split` | Mínimo de muestras para dividir un nodo | 2 |
| `n_jobs` | Núcleos de CPU para paralelizar | -1 (todos) |
| `oob_score` | Calcular score Out-of-Bag | True |

### 4.1 — Entrenar el Random Forest

A diferencia del Árbol de Decisión, Random Forest generalmente **no necesita búsqueda de hiperparámetros** tan exhaustiva porque el promediado de árboles ya controla el overfitting naturalmente.

In [ ]:
# Instancia el Random Forest con 200 árboles
# n_estimators=200: cuántos árboles construirá el bosque (más = más estable, pero más lento)
# max_features='sqrt': en cada nodo considera sqrt(14) ≈ 3-4 features aleatorias
# oob_score=True: calcula el score Out-of-Bag automáticamente (validación gratis)
# n_jobs=-1: usa todos los núcleos disponibles para paralelizar el entrenamiento
# random_state=42: semilla para reproducibilidad del muestreo aleatorio
rf = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

# .fit() entrena los 200 árboles en paralelo, cada uno con su muestra bootstrap
# Internamente, cada árbol ve aproximadamente el 63.2% de los datos de entrenamiento
# (el resto es su muestra OOB usada para la estimación del error OOB)
rf.fit(X_train, y_train)

# .predict() hace que cada árbol vote y devuelve la clase más votada por cada estudiante
y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)

# oob_score_: estimación del accuracy calculada con los datos OOB de cada árbol
# Es una estimación de generalización sin usar el conjunto de prueba
print(f'Accuracy Random Forest en PRUEBA: {acc_rf:.1%}')
print(f'Accuracy OOB (estimación interna): {rf.oob_score_:.1%}')
print()
print('OOB ≈ Test accuracy → el modelo generaliza bien y no hay data leakage.')

### 4.2 — ¿Cuántos árboles necesito?

La accuracy del Random Forest mejora con más árboles hasta cierto punto, donde se estabiliza.
Este gráfico muestra el **punto de convergencia**: agregar más árboles después de ese punto no mejora el modelo.

In [ ]:
# Evaluamos el accuracy en prueba para diferentes números de árboles
n_arboles_lista = [1, 5, 10, 25, 50, 100, 150, 200, 300, 500]
acc_por_n = []

for n in n_arboles_lista:
    # Entrena un Random Forest con n árboles
    rf_n = RandomForestClassifier(n_estimators=n, max_features='sqrt',
                                   n_jobs=-1, random_state=42)
    rf_n.fit(X_train, y_train)

    # Calcula accuracy en prueba y la guarda en la lista
    acc_por_n.append(accuracy_score(y_test, rf_n.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(n_arboles_lista, acc_por_n, marker='o', color='seagreen', linewidth=2)
ax.axhline(acc_arbol, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Árbol único (acc={acc_arbol:.1%})')   # línea de referencia del árbol solo

ax.set_title('Accuracy vs Número de árboles en el Random Forest
'
             'La curva se estabiliza: más árboles después del codo = costo sin ganancia')
ax.set_xlabel('Número de árboles (n_estimators)')
ax.set_ylabel('Accuracy en prueba')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 — Importancia de features del Random Forest

La importancia de Random Forest es **más confiable** que la del árbol individual porque promedia la reducción de Gini sobre los 200 árboles, reduciendo el sesgo hacia features con muchos valores únicos.

Además, con 200 árboles usando muestras aleatorias de features, todas las features tienen oportunidad de aparecer — no solo las más dominantes.

In [ ]:
# feature_importances_ en Random Forest es el promedio de la importancia
# de esa feature a través de los 200 árboles del bosque
importancias_rf = pd.DataFrame({
    'feature':     features,
    'importancia': rf.feature_importances_   # promedio de los 200 árboles
}).sort_values('importancia', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))

# Colorea en naranja para diferenciar visualmente del árbol simple (azul)
ax.barh(importancias_rf['feature'][::-1], importancias_rf['importancia'][::-1],
        color='darkorange', edgecolor='white')

ax.set_title('Importancia de features — Random Forest, (promedio de 200 árboles, más robusto)')
ax.set_xlabel('Importancia media (reducción de Gini promediada)')
plt.tight_layout()
plt.show()

---
## ⚖️ Sección 5 — Comparación de los tres modelos

### ¿Cuándo usar cada modelo?

Ahora que entrenamos tres clasificadores diferentes sobre el mismo dataset y el mismo conjunto de prueba, podemos compararlos de forma justa.

**Principio de comparación justa:**
- Mismo `random_state=42` en todos los splits
- Mismo `X_test`, `y_test` para evaluar
- Mismas features para todos los modelos

Esta sección responde: **¿qué modelo elegirías para producción y por qué?**

In [ ]:
# ── Reentrenar Regresión Logística para comparación justa ────────────────────
# Usamos el MISMO X_train y X_test que los árboles para que la comparación sea válida
# La Logística necesita escalado previo; los árboles NO (son insensibles a la escala)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # escala features del entrenamiento
X_test_sc  = scaler.transform(X_test)        # aplica los mismos parámetros al test

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_sc, y_train)             # entrena sobre datos escalados
y_pred_lr = log_reg.predict(X_test_sc)       # predice sobre test escalado

# ── Tabla comparativa de accuracy ────────────────────────────────────────────
resultados = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),     # accuracy de la regresión logística
        accuracy_score(y_test, y_pred_arbol),  # accuracy del árbol de decisión
        accuracy_score(y_test, y_pred_rf)      # accuracy del random forest
    ],
    'Necesita escalado': ['Sí', 'No', 'No'],
    'Interpretable':     ['Media', 'Alta', 'Baja'],
    'Riesgo overfitting':['Bajo', 'Alto', 'Bajo']
})

# Formatea la columna de accuracy como porcentaje con 2 decimales
resultados['Accuracy'] = resultados['Accuracy'].map('{:.1%}'.format)
resultados

### 5.1 — Matriz de confusión comparada

Comparamos las matrices de confusión de los tres modelos para ver si cometen errores en los mismos estudiantes o en diferentes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Datos de las tres matrices: (predicciones, título, color)
modelos_plot = [
    (y_pred_lr,    'Regresión Logística', 'Blues'),
    (y_pred_arbol, f'Árbol (depth={mejor_depth})', 'Greens'),
    (y_pred_rf,    'Random Forest (200 árboles)', 'Oranges')
]

for ax, (y_pred, titulo, cmap) in zip(axes, modelos_plot):
    cm = confusion_matrix(y_test, y_pred)         # calcula la matriz 2×2

    # ConfusionMatrixDisplay visualiza la matriz con colores y anotaciones automáticas
    disp = ConfusionMatrixDisplay(cm, display_labels=['Reprueba', 'Aprueba'])
    disp.plot(cmap=cmap, ax=ax, colorbar=False)   # colorbar=False: más limpio
    ax.set_title(titulo, fontsize=11)

fig.suptitle('Comparación de matrices de confusión — mismo conjunto de prueba (200 estudiantes)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 5.2 — Reporte completo por clase

La accuracy sola puede ser engañosa. El reporte completo muestra Precision, Recall y F1 para cada clase por separado, lo cual es esencial cuando el costo de los errores no es simétrico.

> **En este contexto educativo:** un Falso Negativo (predecir que reprueba cuando en realidad aprueba) puede ser tan dañino como un Falso Positivo. Pero en otros contextos (detección de fraude, diagnóstico médico) el Recall de la clase positiva es crítico.

In [ ]:
# Imprimimos el reporte completo del modelo con mejor accuracy (Random Forest)
print('=== Reporte completo — Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Reprueba (0)', 'Aprueba (1)']))

### 5.3 — Importancia de features: Árbol vs Random Forest

Comparar las importancias de ambos modelos revela si el árbol individual "concentró" demasiado en pocas features (lo cual puede ser señal de overfitting de la selección de features).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Árbol de decisión ─────────────────────────────────────────────────────────
# Solo mostramos features con importancia > 0 (el árbol limitado no usa todas)
imp_arbol = importancias_arbol[importancias_arbol['importancia'] > 0]
axes[0].barh(imp_arbol['feature'][::-1], imp_arbol['importancia'][::-1],
             color='steelblue', edgecolor='white')
axes[0].set_title(f'Árbol de Decisión (depth={mejor_depth})
{len(imp_arbol)} features usadas de {len(features)}')
axes[0].set_xlabel('Importancia')

# ── Random Forest ─────────────────────────────────────────────────────────────
# El RF siempre tiene importancia > 0 para casi todas las features
axes[1].barh(importancias_rf['feature'][::-1], importancias_rf['importancia'][::-1],
             color='darkorange', edgecolor='white')
axes[1].set_title(f'Random Forest (200 árboles)
{len(features)} features con importancia > 0')
axes[1].set_xlabel('Importancia promedio')

fig.suptitle('Importancia de features: Árbol vs Random Forest
'
             'El RF distribuye más la importancia entre features (menos sesgado)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 🔄 Sección 6 — Validación cruzada (Cross-Validation)

### ¿Por qué necesitamos validación cruzada?

Hasta ahora evaluamos todos los modelos sobre el **mismo conjunto de prueba de 200 estudiantes**.
Esto introduce un riesgo: que el conjunto de prueba tenga características inusuales que favorezcan artificialmente a un modelo.

La **validación cruzada k-fold** repite el proceso de evaluación K veces, usando cada vez una partición diferente como conjunto de prueba:

```
Dataset completo (1000 estudiantes)

Fold 1: [PRUEBA  | TRAIN | TRAIN | TRAIN | TRAIN]
Fold 2: [TRAIN  | PRUEBA | TRAIN | TRAIN | TRAIN]
Fold 3: [TRAIN  | TRAIN | PRUEBA | TRAIN | TRAIN]
Fold 4: [TRAIN  | TRAIN | TRAIN | PRUEBA | TRAIN]
Fold 5: [TRAIN  | TRAIN | TRAIN | TRAIN | PRUEBA]
         ──────────────────────────────────────────
Resultado final: promedio de los 5 accuracies ± desviación estándar
```

El resultado es una **estimación más robusta** del rendimiento real porque promedia sobre 5 particiones diferentes.

### ¿Qué dice la desviación estándar?

- **Std pequeña** → el modelo es estable: se comporta de forma similar sin importar qué datos use
- **Std grande** → el modelo es inestable: muy sensible a los datos específicos de entrenamiento

In [ ]:
# Definimos los modelos a comparar con sus hiperparámetros finales
modelos_cv = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Árbol de Decisión':   DecisionTreeClassifier(max_depth=mejor_depth, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
}

resultados_cv = {}

for nombre, modelo in modelos_cv.items():
    # cross_val_score realiza la validación cruzada completa en una sola llamada
    # cv=5: divide el dataset en 5 folds y repite el proceso 5 veces
    # scoring='accuracy': métrica a calcular en cada fold
    # X (sin escalar) para árboles; la Logística idealmente necesitaría un pipeline con scaler
    # Para esta comparación usamos X sin escalar: la diferencia en Logística es pequeña aquí
    scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy', n_jobs=-1)
    resultados_cv[nombre] = scores   # guarda el array de 5 accuracies

# Construye tabla resumen con media y desviación estándar de los 5 folds
resumen_cv = pd.DataFrame({
    'Modelo': list(resultados_cv.keys()),
    'Accuracy media (5-fold)': [s.mean() for s in resultados_cv.values()],
    'Desv. estándar':          [s.std()  for s in resultados_cv.values()]
}).round(4)

resumen_cv['Accuracy media (5-fold)'] = resumen_cv['Accuracy media (5-fold)'].map('{:.1%}'.format)
resumen_cv['Desv. estándar']          = resumen_cv['Desv. estándar'].map('{:.4f}'.format)
resumen_cv

In [ ]:
# Visualización: boxplot de las 5 accuracies por modelo
# El boxplot muestra la mediana, la dispersión y si hay outliers entre los folds
fig, ax = plt.subplots(figsize=(8, 4))

# Prepara los datos para el boxplot: lista de arrays de scores
datos_box = [resultados_cv[n] for n in modelos_cv.keys()]

bp = ax.boxplot(datos_box, labels=modelos_cv.keys(),
                patch_artist=True,   # rellena las cajas con color
                widths=0.5)

# Colorea cada caja de forma diferente para distinguir los modelos
colores_box = ['#5c9ee0', '#5fad56', '#ff7f0e']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Validación cruzada 5-fold — distribución de accuracy por fold
'
             'Caja más angosta = modelo más estable entre particiones')
ax.set_ylabel('Accuracy')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 🏋️ Sección 7 — Ejercicio resuelto

### Planteamiento

Sabemos que `study_hours_per_day` es con diferencia la feature más importante.
El árbol de decisión probablemente la usa como su primera pregunta (la raíz).

**Tu tarea:**
1. Entrena un Árbol de Decisión con `max_depth=1` (solo una pregunta, la más importante)
2. Imprime qué pregunta hace en el nodo raíz y cuál es su umbral
3. Calcula la accuracy con solo esa regla
4. Compara con el modelo completo y reflexiona: ¿cuánta información agrega el resto de features?

In [ ]:
# ── PASO 1: Árbol con una sola pregunta ─────────────────────────────────────
# max_depth=1 significa: el árbol hace UNA sola pregunta y luego predice
arbol_1 = DecisionTreeClassifier(max_depth=1, criterion='gini', random_state=42)
arbol_1.fit(X_train, y_train)    # aprende cuál es la pregunta más importante

# ── PASO 2: Identificar la pregunta del nodo raíz ────────────────────────────
# tree_.feature[0] es el índice de la feature usada en el nodo 0 (la raíz)
# tree_.threshold[0] es el umbral de esa feature en el nodo raíz
idx_feature_raiz = arbol_1.tree_.feature[0]      # índice numérico de la feature
umbral_raiz      = arbol_1.tree_.threshold[0]    # valor umbral de la pregunta

print(f'Pregunta del nodo raíz:')
print(f'  ¿{features[idx_feature_raiz]} <= {umbral_raiz:.3f}?')
print()

# ── PASO 3: Accuracy con una sola regla ──────────────────────────────────────
acc_1 = accuracy_score(y_test, arbol_1.predict(X_test))
print(f'Accuracy con 1 pregunta (depth=1):         {acc_1:.1%}')
print(f'Accuracy árbol óptimo (depth={mejor_depth}):          {acc_arbol:.1%}')
print(f'Accuracy Random Forest (200 árboles):      {acc_rf:.1%}')
print()

# ── PASO 4: Reflexión ─────────────────────────────────────────────────────────
ganancia = (acc_arbol - acc_1) * 100
print(f'Ganancia del árbol óptimo sobre depth=1:   +{ganancia:.1f} puntos porcentuales')
print()
print('Interpretación:')
print(f'  Una sola regla sobre study_hours_per_day ya captura buena parte')
print(f'  del patrón. Las features adicionales aportan, pero study_hours')
print(f'  sola es responsable de la mayor parte de la predicción.')

---
## 🏁 Resumen de la clase

### Lo que construiste hoy

```
Dataset preparado (clase anterior)
    │
    ├─ ÁRBOL DE DECISIÓN ──────────────────────────────────────────────────────
    │    ├── DecisionTreeClassifier(criterion='gini')
    │    ├── Efecto del overfitting sin max_depth
    │    ├── Búsqueda del max_depth óptimo (gráfico train vs test)
    │    ├── Visualización con plot_tree()
    │    ├── Reglas en texto con export_text()
    │    └── Importancia de features (feature_importances_)
    │
    ├─ RANDOM FOREST ──────────────────────────────────────────────────────────
    │    ├── RandomForestClassifier(n_estimators=200, oob_score=True)
    │    ├── Bootstrap + selección aleatoria de features por nodo
    │    ├── OOB score como estimación interna de generalización
    │    ├── Convergencia: accuracy vs número de árboles
    │    └── Importancia de features más robusta
    │
    ├─ COMPARACIÓN DE MODELOS ─────────────────────────────────────────────────
    │    ├── Tabla accuracy: Logística vs Árbol vs Random Forest
    │    ├── Matrices de confusión comparadas
    │    └── Importancia de features: árbol vs ensemble
    │
    └─ VALIDACIÓN CRUZADA ─────────────────────────────────────────────────────
         ├── cross_val_score(modelo, X, y, cv=5)
         ├── Media ± desviación estándar de los 5 folds
         └── Boxplot de estabilidad entre folds
```

---

### Diferencias clave entre los modelos de hoy

| | Árbol de Decisión | Random Forest |
|---|---|---|
| **Número de modelos** | 1 árbol | N árboles (ensemble) |
| **Overfitting** | Alto sin max_depth | Bajo (se cancela entre árboles) |
| **Interpretabilidad** | Muy alta (puedes ver el árbol) | Baja (200 árboles juntos) |
| **Velocidad** | Muy rápido | Más lento (pero paralelizable) |
| **Necesita escalado** | No | No |
| **Importancia features** | Puede sesgarse | Más robusta y distribuida |
| **Parámetro clave** | `max_depth` | `n_estimators` |

---

### 🧠 Preguntas para reflexionar

1. El árbol sin `max_depth` tiene accuracy perfecta en entrenamiento pero peor en prueba. ¿Por qué ocurre esto y cómo lo llamamos?
2. ¿Por qué Random Forest no necesita que escales las features con StandardScaler mientras que la Regresión Logística sí?
3. En el ejercicio, `study_hours_per_day` fue la primera pregunta del árbol con `depth=1`. ¿Eso significa que es la única variable que importa para predecir si un estudiante aprueba?
4. La validación cruzada dio una **desviación estándar** para cada modelo. ¿Qué implica una std alta para el desempeño del modelo en producción?
5. Random Forest tiene mayor accuracy que el árbol individual, pero es mucho menos interpretable. ¿En qué situación preferirías el árbol a pesar de tener menos accuracy?